<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex08.1-stationary-heat/Ex08.1_heatsink_new.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the library files from the public course repository, so nothing needs uploading.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Raissi, Perdikaris & Karniadakis, *Physics-informed neural networks*, J. Comput. Phys. 378 (2019) 686–707.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_08.1 — A Power Module on a Finned Heat Sink

**Deep Learning for Engineering · Aalborg University · Part 2 · paired with L8.1 · Stationary Heat**

A power module sits on an aluminium heat sink and dissipates 150 W. The heat
spreads through the base plate and leaves through ten fins into forced air. How
hot does the module's footprint get, and how far above what a hand calculation
says? It is steady conduction, the Poisson equation of L7 with real units, on a
rectangle whose solution is **exact**: a cosine series in elementary functions.
That is the point of this version of the exercise. Finite differences on a
grid and a physics-informed network are both scored against the truth, and
their times are compared on what runs online.

| | what you do | TODO |
|---|---|---|
| **1 – 3** | the problem, its data and its physics: the fins, the base plate, the exact solution | — |
| **4** | finite differences on a grid, refined until an agreed 0.01 K | — |
| **5** | a PINN with the hand calculation built in, and the comparison | the trial function, the residual |
| **6** | what the comparison says | — |
| **7 – 8** | your report, and the mini project proposal | your answers |

About five minutes on a CPU, most of it the network's training. Numbers are
printed with at most two decimals.

---

## 0 · Setup

In [ ]:
# files-cell v2 ----------------------------------------------------------
# The library files must sit beside the notebook. Locally they already do. On
# Google Colab they are fetched from the public course repository on every
# run, so a file left over from an earlier session cannot shadow it.
import os, sys, urllib.request
FILES = ['course_core.py', 'pinn_core.py', 'heatsink.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex08.1-stationary-heat/"
ON_COLAB = "google.colab" in sys.modules
for f in FILES:
    if ON_COLAB or not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
for _m in ("course_core", "pinn_core", "heatsink"):
    sys.modules.pop(_m, None)
print("files ready:", ", ".join(FILES))

In [ ]:
# --- setup ---------------------------------------------------------------
import os, time
for f in ("course_core.py", "pinn_core.py", "heatsink.py"):
    assert os.path.exists(f), f"{f} is missing - run the files cell above first"
from pinn_core import *                                  # noqa: F401,F403
import course_core as cc
import heatsink as hs
import numpy as np, torch, matplotlib.pyplot as plt
import scipy.sparse as sp, scipy.sparse.linalg as spla

cc.OUTPUT_DIR = "Ex08.1_outputs"                              # where the report is written
set_seed(88)  # fix the random start so a run repeats
print("torch", torch.__version__, " device:", DEVICE, " dtype:", torch.get_default_dtype())

def middle(fn, reps=5):                                    # the middle of five timings, after a warm-up
    fn(); ts = []
    for _ in range(reps):
        t0 = time.perf_counter(); fn(); ts.append(time.perf_counter() - t0)
    return float(np.median(ts))

def online(fn, reps=5):                                    # the same, with the GPU made to finish before the clock stops
    def run():
        fn()
        if DEVICE.type == "cuda": torch.cuda.synchronize()
    return middle(run, reps)

---

## 1 · The problem

An extruded aluminium heat sink: a base plate 100 mm wide and 10 mm thick with
ten straight fins, 2 mm thick and 40 mm long, at a 10 mm pitch. Forced air at
40 °C flows along the fins. A power module 30 mm wide sits in the middle of
the base plate and dissipates 150 W per 100 mm of depth, evenly over its
footprint. The ends of the base plate are insulated, and so is the top face
outside the module. What is the temperature at every point of the base plate,
and how hot is the hottest point under the module?

## 2 · The data

| | |
|---|---|
| base plate | 100 × 10 mm, aluminium alloy 6061, $k$ = 167 W/(m·K) |
| fins | 10 straight fins, 2 × 40 mm, pitch 10 mm |
| air | forced, 40 °C, $h$ = 40 W/(m²K) on the fins and the bare base — *typical value* |
| module | 150 W on a 30 mm wide footprint, centred; $q''$ = 50 kW/m² — *typical values* |
| elsewhere | the ends of the base and the top face outside the module are insulated |

The numbers are of the right order for a small drive and were not measured on
any product; every result below depends on them.

## 3 · The physics

**The fins enter exactly.** A straight fin is the first problem of L8.1 with
an exact solution: conduction along it, convection from both faces, and the
textbook efficiency

$$\eta = \frac{\tanh(mL_c)}{mL_c}, \qquad m = \sqrt{\frac{2h}{k\,t_f}}, \qquad L_c = L_f + \frac{t_f}{2}$$

(Incropera ch. 3). Per pitch $p$ of base the fin and the bare base beside it
take $h\,(\eta\, 2L_c + p - t_f)$ watts per kelvin and metre of depth, so the
finned face acts on the base plate as **one effective coefficient**

$$h_{\mathrm{eff}} = h\,\frac{\eta\,2L_c + p - t_f}{p}$$

That is how a heat sink is sized by hand, and it is what turns the base plate
into a rectangle. (How good the collapse is, with the fins resolved, is mini
project MP8.1A.)

**The base plate** is steady conduction with all three kinds of wall of
L8.1's *Thermal Boundary Conditions*: a **flux** on the module's footprint,
**convection** on the fin side, and **insulated** ends.

$$k\,(T_{xx} + T_{zz}) = 0, \qquad
k\,T_z = q''(x)\ \text{at}\ z = t_b, \qquad
k\,T_z = h_{\mathrm{eff}}\,(T - T_\infty)\ \text{at}\ z = 0, \qquad
T_x = 0\ \text{at}\ x = 0, W$$

No wall holds a temperature. The level is fixed by the convection: the heat
that enters must leave, and leaves only at a rate $h_{\mathrm{eff}}$ times the
rise. **The hand calculation** uses exactly that: spread the 150 W over the
whole base, $q_{\mathrm{mean}}$ = 15 kW/m², and the fin side sits
$q_{\mathrm{mean}}/h_{\mathrm{eff}}$ above the air, the module side another
$q_{\mathrm{mean}}\,t_b/k$ higher. It is one dimension, and it misses the
**spreading**: the heat enters on 30 mm and leaves over 100 mm, so under the
module the base is hotter than the mean. The two-dimensional solution is what
says by how much.

**Scaled units.** $X = x/W$, $Z = z/t_b$, and the rise above the air over
$\Theta = q_{\mathrm{mean}}/h_{\mathrm{eff}}$, the hand calculation's own
number, so that $\theta = 1$ means "what the hand calculation says". The base
plate becomes the unit square and

$$\varepsilon^2\,\theta_{XX} + \theta_{ZZ} = 0, \qquad
\theta_Z = \mathrm{Bi}\,g(X)\ \text{at}\ Z = 1, \qquad
\theta_Z = \mathrm{Bi}\,\theta\ \text{at}\ Z = 0, \qquad
\theta_X = 0\ \text{at}\ X = 0, 1$$

with $\varepsilon = t_b/W$ = 0.1 the aspect ratio, $\mathrm{Bi} =
h_{\mathrm{eff}} t_b/k$ the Biot number of the base plate, and
$g = q''/q_{\mathrm{mean}}$ = 3.33 on the footprint and 0 elsewhere.

**The exact solution.** Insulated ends make cosines in $x$ exact, and each
cosine has its own profile through the thickness:

$$\theta(x, z) = \sum_{n \ge 0} \cos\!\left(\frac{n\pi x}{W}\right) Z_n(z),
\qquad Z_0 \text{ linear}, \qquad Z_n = \frac{q_n}{k\lambda_n}\,
\frac{\cosh\lambda_n z + r_n \sinh\lambda_n z}{\sinh\lambda_n t_b + r_n\cosh\lambda_n t_b}$$

with $\lambda_n = n\pi/W$, $r_n = h_{\mathrm{eff}}/(k\lambda_n)$ and $q_n$
the cosine coefficients of the module's flux. Elementary functions, no
eigenvalue to find. Where the flux steps, at the footprint's edges, the series
converges slowly, so `hs.exact` sums that part in closed form (a dilogarithm)
and keeps only a remainder that dies exponentially: the field is exact to
rounding and costs milliseconds. **It is the truth** that both methods below
are scored against. The cell prints the data, draws the heat sink, and plots
the exact field.

In [ ]:
hs.describe_problem()
fig, ax = plt.subplots(figsize=(9.5, 4.6))
hs.draw_heatsink(ax)                                         # the cross-section, with the domain that is solved
plt.tight_layout(); plt.show()

xg, zg = np.linspace(0, hs.W_BASE, 401), np.linspace(0, hs.T_BASE, 41)
XG, ZG = np.meshgrid(xg, zg)
T_exact = hs.T_AIR + hs.exact(XG, ZG)                        # the exact temperature on a fine grid, °C
hot = hs.hottest(); hand = hs.THETA + hs.Q_MEAN * hs.T_BASE / hs.K_AL
print(f"hottest point: {hs.T_AIR + hot:.2f} °C, under the module's centre; the hand calculation says {hs.T_AIR + hand:.2f} °C, "
      f"so the spreading adds {hot - hand:.2f} K")
print(f"energy balance of the exact field: {hs.H_EFF * np.trapezoid(hs.exact(xg, np.zeros_like(xg)), xg) * hs.DEPTH:.2f} W leave through the fins of {hs.POWER:.0f} W")
fig, ax = plt.subplots(figsize=(9.5, 2.6))
c = ax.pcolormesh(XG * 1e3, ZG * 1e3, T_exact, cmap="inferno", shading="auto")
ax.plot([hs.MODULE[0] * 1e3, hs.MODULE[1] * 1e3], [hs.T_BASE * 1e3] * 2, color="tab:red", lw=4, label="the module")
ax.set_aspect("equal"); ax.set_xlabel("x  [mm]"); ax.set_ylabel("z  [mm]"); ax.legend(frameon=False, loc="lower right")
plt.colorbar(c, ax=ax, label="exact temperature  [°C]"); plt.title("the base plate: the exact field", fontsize=10)
plt.tight_layout(); plt.show()

**What you should see.** The fin efficiency 0.884 and $h_{\mathrm{eff}}$ =
322 W/(m²K), eight times the bare $h$: that is what the fins are for. The hand
calculation puts the module side at 87.47 °C; the exact solution puts the
hottest point at 92.45 °C, so the spreading adds 4.98 K, about a tenth of the
rise. The exact field sends 150.00 W through the fins. In the picture the
base is nearly uniform through its thickness and warmer under the module: the
heat spreads sideways through 10 mm of aluminium before it finds a fin.

---

## 4 · Finite differences, scored against the exact solution

**The grid fits the geometry.** The base plate is a rectangle, so a grid of
$n_x \times n_z$ nodes lays its nodes on every wall and there is no
staircase. The five-point stencil asks the equation at every interior node;
each wall is handled by a **ghost node** one spacing outside it, eliminated
with the wall's condition:

* insulated end: $\theta_{-1} = \theta_{1}$, a mirrored node;
* module face: $\theta_{N+1} = \theta_{N-1} + 2h_z\,\mathrm{Bi}\,g$, the flux
  enters through the ghost; a node on the footprint's edge carries half the
  flux;
* fin side: $\theta_{-1} = \theta_{1} - 2h_z\,\mathrm{Bi}\,\theta_0$, the
  convection.

One sparse solve gives every node (`hs.fdm_steady`). **The agreed accuracy is
0.01 K**, a tenth of what a thermocouple resolves; the exact solution lets us
ask for that. The cell refines the grid until it is met, and times one solve.

In [ ]:
TARGET = 0.01                                                 # K: the agreed accuracy
XG_s, ZG_s = XG / hs.W_BASE, ZG / hs.T_BASE                   # the scoring grid, scaled
rows_fdm = []
for nx, nz in ((21, 5), (41, 9), (81, 17), (161, 33), (321, 65)):
    X, Z, th = hs.fdm_steady(nx, nz)
    XX, ZZ = np.meshgrid(X * hs.W_BASE, Z * hs.T_BASE)
    err = np.abs(hs.kelvin(th) - hs.exact(XX, ZZ))           # against the exact field, on the grid's own nodes
    sec = online(lambda: hs.fdm_steady(nx, nz), reps=3)
    rows_fdm.append((nx, nz, err.max(), sec))
    i = np.unravel_index(err.argmax(), err.shape)
    print(f"  {nx:3d} x {nz:3d} nodes: worst error {err.max():.2e} K at x = {X[i[1]] * 100:.1f} mm on the {'module' if Z[i[0]] > 0.5 else 'fin'} side, "
          f"one solve {sec * 1e3:.1f} ms" + ("   <- meets 0.01 K" if err.max() <= TARGET else ""))
NX, NZ = 81, 17                                               # the coarsest grid that meets it

**What you should see.** The worst error halving with every halving of the
spacing - 3.73e-02, 1.85e-02, 9.16e-03, 4.54e-03, 2.26e-03 K - and always at
the footprint's edge on the module face, where the flux steps from 50 kW/m² to
nothing and the field has a corner in its gradient. Away from that edge the
grid is second order; at it, first. **81 × 17 nodes meet 0.01 K in a few
milliseconds.** Compare Ex_08.1's plate with a channel, where a staircase wall
cost 641 × 641 nodes for 0.10 K: a rectangle is the grid's home ground.

**Two ways to cover the base plate.** The cell below draws the grid that meets
the target and the points the network of section 5 trains on.

In [ ]:
X, Z = np.linspace(0, 1, NX), np.linspace(0, 1, NZ)
pts = hs.sample_base(2000)                                    # the network's collocation points
top, bottom, left, right = hs.sample_faces(200, 60)          # and its points on the four walls
fig, ax = plt.subplots(2, 1, figsize=(11, 5.2))
for xv in X: ax[0].plot([xv * 100, xv * 100], [0, 10], lw=0.3, color="0.5")
for zv in Z: ax[0].plot([0, 100], [zv * 10, zv * 10], lw=0.3, color="0.5")
ax[0].set_title(f"finite differences: {NX} x {NZ} nodes, every wall on a grid line", fontsize=10)
ax[1].plot(pts[:, 0] * 100, pts[:, 1] * 10, ".", ms=2, color="0.45", label="2000 points: the equation")
ax[1].plot(top[:, 0] * 100, top[:, 1] * 10, ".", ms=4, color="tab:red", label="the module face: the flux, denser at the footprint's edges")
ax[1].plot(bottom[:, 0] * 100, bottom[:, 1] * 10, ".", ms=4, color="tab:blue", label="the fin side: convection")
ax[1].set_title("PINN: scattered points, no mesh; the insulated ends need no points (section 5)", fontsize=10)
for q in ax:
    q.plot([hs.MODULE[0] * 100, hs.MODULE[1] * 100], [10.3, 10.3], color="tab:red", lw=3)
    q.set_aspect("equal"); q.set_xlim(-1, 101); q.set_ylim(-0.6, 11); q.set_ylabel("z  [mm]")
ax[1].set_xlabel("x  [mm]"); ax[1].legend(frameon=False, fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.35), ncol=3)
plt.tight_layout(); plt.show()

**What you should see.** Above, the grid: 1377 nodes, the footprint's edges
on grid lines. Below, the network's points: 2000 scattered in the plate, a
line on each face where a condition is asked, denser on the module face where
the flux steps, and nothing on the two ends.

---

## 5 · A physics-informed network, and the comparison

**Build in what you know; let the network learn what you do not.** The hand
calculation is a solution of the equation and of two of the four walls (the
mean flux, the convection); only the spreading is missing. So the trial
function is

$$\hat\theta(X, Z) = \underbrace{1 + \mathrm{Bi}\,Z}_{\text{the hand calculation}}
\;+\; S\,\mathcal{N}\big(\cos \pi X,\ Z\big)$$

The network $\mathcal{N}$, four layers of 32 tanh neurons, learns the
spreading; $S$ = 0.1 is its size, fixed, not trained, there because the
spreading is a tenth of the rise. Its first input is **$\cos \pi X$ rather
than $X$**: a function of $\cos \pi X$ is even about both ends of the base,
so $\partial\hat\theta/\partial X$ = 0 at $X$ = 0 and $X$ = 1 holds exactly,
for any weights. This is **hard enforcement** of the insulated ends, by a
change of input rather than by a multiplier, and the ends need no loss term.

**The two faces stay loss terms**, with a weight - **soft enforcement**,
because the flux and the convection are conditions on a derivative:
$(\hat\theta_Z/\mathrm{Bi} - g)^2$ on the module face and
$(\hat\theta_Z/\mathrm{Bi} - \hat\theta)^2$ on the fin side. Both are divided
by Bi = 0.019 so that they are of order one, like the residual, which is
divided by Bi for the same reason. **The energy balance is a third loss term**:
the mean of $\hat\theta$ along the fin side must be exactly 1, because the heat
leaving is $h_{\mathrm{eff}}$ times that mean and must equal the 150 W that
enter. With no wall holding a temperature, this is what pins the level; without
it the network drifts by a kelvin.

**TODO 1** is the trial function; **TODO 2** the residual of the scaled
equation, $(\varepsilon^2\hat\theta_{XX} + \hat\theta_{ZZ})/\mathrm{Bi}$.

In [ ]:
SCALE = 0.1                                                   # the size of the spreading over the hand calculation
class BasePlate(torch.nn.Module):                             # a network: layers in __init__, the computation in forward
    def __init__(self):
        super().__init__()
        self.net = MLP(n_in=2, n_hidden=32, n_layers=4)       # dense tanh network: cos(pi X), Z in
    def forward(self, p):
        X, Z = p[:, 0:1], p[:, 1:2]
        # TODO 1: the trial function - the hand calculation 1 + Bi Z, plus SCALE times the network
        # evaluated on the features (cos(pi X), Z), joined with torch.cat along dimension 1.
        #
        #   features = torch.cat([torch.cos(np.pi * X), Z], 1)
        #   return 1.0 + hs.BI * Z + SCALE * self.net(features)
        raise NotImplementedError("Write the trial function")

In [ ]:
# TODO 2: the residual of the scaled equation, (EPS^2 theta_XX + theta_ZZ) / Bi.
#
#   def residual(model, p):
#       th = model(p)
#       return (hs.EPS ** 2 * d2(th, p, 0) + d2(th, p, 1)) / hs.BI
raise NotImplementedError("Write residual(model, p)")

In [ ]:
W_F, W_E = 10.0, 10.0                                         # the weights of the two faces and of the energy balance
P_in = to_tensor(pts, requires_grad=True)                     # collocation points, differentiable
P_top, P_bot = to_tensor(top, requires_grad=True), to_tensor(bottom, requires_grad=True)   # the two faces, differentiable
g_top = to_tensor(hs.footprint(top[:, 0])[:, None])           # q''/q_mean at each point of the module face
P_bal = to_tensor(np.c_[np.linspace(0, 1, 101), np.zeros(101)])   # where the fin-side mean is taken

def face_loss(model):
    flux = grad(model(P_top), P_top)[:, 1:2] / hs.BI - g_top         # the module face: theta_Z = Bi g; grad = autograd
    conv = grad(model(P_bot), P_bot)[:, 1:2] / hs.BI - model(P_bot)  # the fin side: theta_Z = Bi theta
    return flux.pow(2).mean() + conv.pow(2).mean()

def balance_loss(model):
    return (model(P_bal).mean() - 1.0).pow(2)                 # the heat out equals the heat in: the fin-side mean is 1

model = BasePlate().to(DEVICE)                                # four layers of 32
describe(model.net, n_collocation=len(P_in))                  # the parameter count and the collocation-to-parameter ratio
loss_fn = lambda: residual(model, P_in).pow(2).mean() + W_F * face_loss(model) + W_E * balance_loss(model)   # the composite loss
t0 = time.perf_counter()
# Adam first, then L-BFGS: the Part 2 training recipe
history = train_two_stage(model, loss_fn, adam_steps=1000, lbfgs_steps=400, lr=2e-3, report_every=0)
train_time = time.perf_counter() - t0
print(f"trained in {train_time:.0f} s (offline); residual {residual(model, P_in).pow(2).mean().item():.2e}, "
      f"faces {face_loss(model).item():.2e}, balance {balance_loss(model).item():.2e}")

In [ ]:
X, Z, th_fdm = hs.fdm_steady(NX, NZ)                          # the finite-difference answer on the grid that meets 0.01 K
XX, ZZ = np.meshgrid(X, Z)
T_true = hs.exact(XX * hs.W_BASE, ZZ * hs.T_BASE)             # the exact rise on the same nodes
Pn = to_tensor(np.c_[XX.ravel(), ZZ.ravel()])                 # the same nodes for the network
def pinn_field():
    with torch.no_grad():  # no gradient tracking: evaluation only
        return hs.kelvin(to_numpy(model(Pn)).reshape(XX.shape))
T_f, T_p = hs.kelvin(th_fdm), pinn_field()
e_f, e_p = np.abs(T_f - T_true), np.abs(T_p - T_true)
t_fdm = online(lambda: hs.fdm_steady(NX, NZ), reps=3)         # one grid solve, on this runtime
t_pin = online(pinn_field)                                    # one evaluation of the trained network on the same nodes, on this runtime
balance = hs.H_EFF * hs.THETA * np.trapezoid(to_numpy(model(P_bal)).ravel(), np.linspace(0, 1, 101)) * hs.W_BASE * hs.DEPTH

# the time column is what runs ONLINE: a grid solve against an evaluation of the trained network on the same runtime.
# The training is offline, paid once, and reported on its own line.
print(f"{'':30s}{'unknowns':>10s}{'hottest':>11s}{'worst error':>14s}{'time online':>14s}")
print(f"  {'exact series':28s}{'-':>10s}{hs.T_AIR + T_true.max():8.2f} °C{'0 (the truth)':>14s}{'-':>14s}")
print(f"  {'finite differences, ' + str(NX) + ' x ' + str(NZ):28s}{NX * NZ:10d}{hs.T_AIR + T_f.max():8.2f} °C{e_f.max():11.2e} K{t_fdm * 1e3:11.1f} ms")
print(f"  {'PINN, 4 x 32':28s}{parameter_count(model):10d}{hs.T_AIR + T_p.max():8.2f} °C{e_p.max():11.2e} K{t_pin * 1e3:11.1f} ms   (trained offline in {train_time:.0f} s, once)")
print(f"energy balance of the PINN: {balance:.2f} W leave through the fins of {hs.POWER:.0f} W")

fig, axes = plt.subplots(3, 1, figsize=(10, 7.2))
for ax, vals, title, cmap in ((axes[0], hs.T_AIR + T_true, "exact  [°C]", "inferno"),
                              (axes[1], e_f, f"finite differences {NX} x {NZ}: error  [K]", "viridis"),
                              (axes[2], e_p, "PINN: error  [K]", "viridis")):
    c = ax.pcolormesh(XX * 100, ZZ * 10, vals, cmap=cmap, shading="auto")
    ax.set_title(title, fontsize=10); ax.set_aspect("equal"); ax.set_ylabel("z  [mm]"); plt.colorbar(c, ax=ax, shrink=0.9)
axes[2].set_xlabel("x  [mm]")
plt.tight_layout(); plt.savefig(cc.output_path("Ex08.1_report.png"), dpi=150); plt.show()

**What you should see.** All three put the hottest point at 92.45 °C to
within a few hundredths. Finite differences are within 0.009 K of the exact
field everywhere, their error confined to the footprint's edges. The network
is within about 0.3 K: a smooth misfit of one to two tenths over most of the
plate, and its worst at the footprint's edges, where a smooth network cannot
follow the corner in the gradient. Its energy balance is 150 W to a tenth of
a watt, which the loss term made sure of. (The network's numbers vary a
little from run to run; the grid's do not.)

**By accuracy** the order is exact, finite differences (0.009 K), network
(about 0.3 K). **By time online** both are milliseconds: one sparse solve of
1377 unknowns against one evaluation of 3300 weights at 1377 points, the
network a few times faster. Its training, four minutes on a CPU, is paid once
and offline.

---

## 6 · What the comparison says

* **On a rectangle, finite differences are hard to beat.** Every wall lies on
  a grid line, the stencil is second order away from the one corner in the
  gradient, and 0.01 K costs 1377 unknowns and a few milliseconds. The network
  reaches a thirtieth of that accuracy after minutes of training.
* **The exact solution is what made the comparison fair.** Both methods were
  scored against the truth, not against a third numerical answer; the
  footprint's edge shows up as the hard spot of both for the same reason.
* **What the network needed to get there is the lesson.** With four walls
  that carry a flux, a convection or nothing, and no wall holding a
  temperature, a plain network drifts in level by a kelvin. Building in the
  hand calculation, making the ends insulated by the choice of input, and
  asking for the energy balance as a loss term brought it to a few tenths.
  Each of those is physics the engineer already knew.
* **Online, the two are even.** Once trained, the network answers as fast as
  the grid solves. That matters when the answer is wanted many times - for a
  range of powers or footprints, or inside a controller - which is where the
  training is repaid, and in Ex_08.2, where one number of the model is
  unknown and has to be found from a thermocouple.
* **The fin-efficiency collapse is the one modelling step not checked here.**
  Mini project MP8.1A resolves the fins and finds how far the collapsed model
  is from the real cross-section.

---

## 7 · Your report

**What these cells do.** You answer the questions below; the next cell writes `Ex08.1_report.md` with the results of section 5 and your answers, and the cell after it turns the report into a PDF for Moodle.

Each question builds part of an answer to one of L8.1's questions for the oral examination; the arrow says which. Keep each answer to a few sentences, with your numbers. Write your name in `NAME`, and each answer between the triple quotes that follow its question; an answer needs at least eight words.

In [ ]:
# TODO: write your answers. Every string below must be replaced.

NAME = "Your Name"

ANSWERS = {
    "Which of the three mechanisms of L8.1 appear in this heat sink, where does each act, and which one is the field equation that was solved? (-> L8.1 Q1)": """
""",
    "No wall of the base plate holds a temperature. What fixes the level of the solution, how did the hand calculation use that, and what did the exact solution add to it? (-> L8.1 Q2)": """
""",
    "What did the trial function build in, and why does a network that takes cos(pi X) as its input satisfy the insulated ends for any weights? (-> L8.1 Q3)": """
""",
    "The two faces and the energy balance are loss terms with weights. What happened to the level of the field without the energy balance, and what did its weight trade? (-> L8.1 Q4)": """
""",
    "To conclude: for this one heat sink, solved once, which method would you choose, and under what use would the network's training be repaid? (-> L8.1, Exercise slide)": """
""",
}

In [ ]:
MIN_WORDS = 8                                               # an answer is a few sentences, not a word
short = [(i, len(text.split())) for i, text in enumerate(ANSWERS.values(), 1) if len(text.split()) < MIN_WORDS]
if NAME.strip() in ("", "Your Name") or short:
    print("Not ready - the report is written once these are fixed:")
    if NAME.strip() in ("", "Your Name"):
        print("  NAME: write your name")
    for i, n in short:
        print(f"  answer {i}: {n} word(s); write at least {MIN_WORDS}")
else:
    rows = ["| method | unknowns | hottest point | worst error against the exact solution | time online |", "|---|---|---|---|---|",
            f"| exact series | - | {hs.T_AIR + T_true.max():.2f} °C | 0 (the truth) | - |",
            f"| finite differences, {NX} x {NZ} | {NX * NZ} | {hs.T_AIR + T_f.max():.2f} °C | {e_f.max():.2e} K | {t_fdm * 1e3:.1f} ms |",
            f"| PINN, 4 x 32 | {parameter_count(model)} | {hs.T_AIR + T_p.max():.2f} °C | {e_p.max():.2e} K | {t_pin * 1e3:.1f} ms (trained offline in {train_time:.0f} s) |",
            "", f"Energy balance of the PINN: {balance:.2f} W of {hs.POWER:.0f} W leave through the fins."]
    out = ["# Ex_08.1 — A Power Module on a Finned Heat Sink", "", f"**{NAME}**", "", "Deep Learning for Engineering · Aalborg University · 2026", "",
           "## Results", "", *rows, ""]
    for qq, text in ANSWERS.items():
        out += [f"## {qq}", "", text.strip(), ""]
    path = cc.output_path("Ex08.1_report.md")
    with open(path, "w", encoding="utf-8") as fh:
        fh.write("\n".join(out))
    print("written", path)

**What you should see.** `Not ready` until every answer is yours; then the path of the report. Then run the cell below: it turns the report into a PDF, with the figure of section 5, and downloads it. **Upload the PDF.**

In [ ]:
# Report as PDF for Moodle -----------------------------------------------
# Runs after the report cell above: turns Ex08.1_report.md into Ex08.1_report.pdf, with any figure
# saved as Ex08.1_report*.png embedded above the answers, and downloads it. Upload
# the PDF to Moodle; the .md stays as the source.
import subprocess, sys, glob, os
pdf_path = os.path.join(cc.OUTPUT_DIR, "Ex08.1_report.pdf")
try:
    import markdown, weasyprint
except ImportError:                       # installed already on a second run
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "markdown", "weasyprint"])
    import markdown, weasyprint

md_path = os.path.join(cc.OUTPUT_DIR, "Ex08.1_report.md")
if not os.path.exists(md_path):
    print("No report yet: it is written once every answer above is filled in.\n"
          "Fill them in, run the cell that writes the report, then run this one again.")
else:
    md = open(md_path, encoding="utf-8").read()
    figs = sorted(glob.glob("Ex08.1_report*.png")
                  + glob.glob(os.path.join(cc.OUTPUT_DIR, "Ex08.1_report*.png")))
    if figs:
        imgs = "\n\n".join(f"![{os.path.basename(p)}]({p})" for p in figs)
        i = md.find("\n## ", md.find("## Results") + 1) if "## Results" in md else -1
        md = (md[:i] + "\n\n" + imgs + "\n" + md[i:]) if i > 0 else md + "\n\n" + imgs + "\n"

    html = markdown.markdown(md, extensions=["fenced_code", "tables"])
    css = """body{font-family:Helvetica,Arial,sans-serif;font-size:11pt;margin:2cm}
    h1{font-size:18pt} h2{font-size:13pt;margin-top:18pt}
    pre{background:#f3f4f6;padding:8px;font-size:9.5pt} img{max-width:100%}"""
    weasyprint.HTML(string=f"<html><head><meta charset='utf-8'><style>{css}</style></head>"
                           f"<body>{html}</body></html>", base_url=".").write_pdf(pdf_path)
    print("written", pdf_path, f"with {len(figs)} figure(s)" if figs else "")
    try:
        from google.colab import files
        files.download(pdf_path)
    except ImportError:
        pass

---

## 8 · Mini project proposal

Two problems grown out of this set, for you to take further on your own. Each
student chooses **one** mini project from the Part 2 sets and has **one month**
to solve it individually. The dates and the hand-in are given in L13.

The ground truth is **given to you**, built by the course with a classical
solver and shown to have converged; the worked example below each project is
its first case. You do not solve the physics a second way: the month goes into
the deep learning - making a network predict what the ground truth says, as
accurately and as fast as the requirements ask.

### MP8.1A · The fins resolved

**The problem.** The notebook collapsed ten fins into one coefficient on the fin side, by the
straight-fin efficiency. Resolve them: solve conduction in the whole
cross-section, base plate and fins, with $h$ = 40 W/(m²K) on every face in
the air, and find how far the collapsed model is from the real thing.

| | |
|---|---|
| section | the heat sink of this notebook: base 100 × 10 mm, ten fins 2 × 40 mm at 10 mm pitch |
| module | 150 W on 30 mm of the top face |
| air | 40 °C, $h$ = 40 W/(m²K) on the fin faces, the fin tips and the bare base between the fins |

**Why a plain PINN is not enough.** The domain is a comb. A single network over a bounding box would have to be
told where the metal is; a Robin condition on every one of the thirty fin faces is
thirty loss terms, or one term on points sampled over all of them by length.

**What you build.** A PINN for $T(x, z)$ over the comb, points sampled in the metal only, the
convection on every air face as one weighted loss term over points sampled by
length, the module's flux on the top, scaled so every term is of order one.

**The ground truth you get.** `MP8.1A_new_truth.npz`, from `tools/miniprojects/heatsink_truth.py`: finite
volumes on square cells of 0.25 mm that the rectilinear geometry fits exactly,
28 800 cells, converged to a few hundredths of a kelvin.

**Worked example.** 

| cells | hottest point | fin-side mean |
|---|---|---|
| 1.00 mm | 51.64 K | 45.89 K |
| 0.50 mm | 51.67 K | 45.82 K |
| 0.25 mm | 51.68 K | 45.80 K |
| the notebook's collapsed model | 52.45 K | 46.57 K |

The collapsed model is 0.77 K warm at the hottest point, 1.5 % of the rise: conservative, because the fin efficiency smears the fins' heat over the whole pitch where the real base is cooler above a fin root.

**Requirements.**

* The hottest point within **0.1 K** of the ground truth.
* The temperature within **0.2 K** everywhere in the metal.
* The heat through the air faces within **1 %** of 150 W.

**What you hand in.** The error map over the comb, the temperature along one fin against the ground truth and against the straight-fin formula, and one paragraph on where the collapsed model is wrong and why.

### MP8.1B · Two modules on one sink

**The problem.** Split the 150 W between two modules of 15 mm each, placed symmetrically with a
gap between them. How far apart should they sit? Train one network for every
gap from 0 to 40 mm, then find by a gradient through it the gap at which the
hottest point is lowest - and say why the answer is not simply "as far as possible".

| | |
|---|---|
| base plate, fins, air | as in this notebook |
| modules | two of 75 W on 15 mm each, symmetric about the centre, gap $d$ from 0 to 40 mm |

**Why a plain PINN is not enough.** A third input, the gap, and a flux pattern that moves with it: the trial
function must carry the hand calculation for every gap, and the flux loss must
be sampled where the footprints are for each one.

**What you build.** A PINN for $\theta(X, Z, d)$ with the hand calculation built in, the two faces
and the energy balance as loss terms, trained once over the range of gaps; then
`torch.autograd` of the hottest point with respect to $d$.

**The ground truth you get.** `MP8.1B_new_truth.npz`, from `tools/miniprojects/heatsink_truth.py`: the exact
series by superposition of the two footprints, for every gap.

**Worked example.** 

| gap | hottest point | where |
|---|---|---|
| 0 mm | 52.45 K | 50.0 mm |
| 10 mm | 50.75 K | 41.2 mm |
| 20 mm | 49.76 K | 34.8 mm |
| 30 mm | 49.28 K | 28.3 mm |

Separating the modules lowers the hottest point, but the gain shrinks as each module nears an insulated end.

**Requirements.**

* The hottest point within **0.1 K** of the exact value at every gap.
* The best gap within **2 mm** of the exact optimum.
* One training for the whole range, no retraining per gap.

**What you hand in.** The hottest point against the gap, network against exact; the gap found by the gradient; and a paragraph on the two effects that compete.